# Chapter 12: Ordered Geometry

Source orientation: printed pages 175-190; PDF pages 193-208. I inspected the rendered source span privately for section structure and terminology: extracting affine and absolute geometries from Euclid, intermediacy, segments/rays/lines, Pasch-style triangle crossing, Sylvester's problem of collinear points, planes and hyperplanes, continuity, and parallelism. This notebook is original teaching material and does not reproduce textbook prose, exercises, screenshots, page crops, or source figures.

## Chapter Goal

Ordered geometry is the part of geometry that can be built before measuring length, angle, area, or curvature. Its primitive information is not "how far" or "how large"; it is "which point lies between which other two points" and "which side of a cut contains which points." The chapter is foundational because it asks how much geometry survives when we keep only incidence and order.

The notebook treats order as executable structure. A betweenness predicate becomes a small truth table. A segment, ray, and line become partitions of a one-dimensional coordinate. Pasch's axiom becomes a crossing experiment in a triangle. Sylvester's theorem becomes a finite search for ordinary lines, lines containing exactly two of a finite non-collinear set. Planes and hyperplanes become region-counting objects. Dedekind continuity becomes a cut that rationals can approach without filling. Parallelism becomes a separation statement about rays from a point relative to a line.

The visual thread is not metric beauty. It is logical control: each diagram exposes a relation that can be checked with signs, determinants, combinations, or finite searches. The goal is for the reader to see why ordered geometry is a common nucleus for later affine, projective, absolute, and hyperbolic chapters.

## Visualization Storyboard

| Section | Representation | Artifact | Inspection target | Check |
|---|---|---|---|---|
| Extracting geometries | overlap/dependency diagram | `euclid-extraction-order-kernel.png` | order is common to affine and absolute reasoning | concept table |
| Intermediacy | number-line model of `[ABC]`, segments, rays, and line | `betweenness-segment-ray-decomposition.png` | segment/ray/line definitions from betweenness | truth table of ordered triples |
| Pasch mechanism | triangle plus a crossing line | `pasch-triangle-crossing.png` | a line entering two sides forces an ordered intersection relation | barycentric and segment-parameter checks |
| Sylvester problem | finite point set and ordinary lines | `sylvester-ordinary-line-search.png` | a non-collinear finite set has a line through exactly two points | exhaustive pair-line search |
| Planes and hyperplanes | region-count formulas and 3D simplex view | `hyperplane-region-counts.png`, `simplex-space-incidence.html` | arrangements decompose space into convex regions | recurrence `f(n,m)=f(n,m-1)+f(n-1,m-1)` |
| Continuity | rational cut around `sqrt(2)` | `dedekind-cut-continuity.png` | a cut can have no rational endpoint and still determine a real boundary | lower/upper rational bounds |
| Parallelism | rays from a point relative to a line | `parallelism-separating-rays.png` | the parallel directions separate meeting rays from nonmeeting rays | ray/line intersection classification |

Matplotlib handles the static proof diagrams because labels, arrows, and exact side-by-side comparisons matter. Plotly is used only for the 3D simplex incidence view, where rotation makes the tetrahedral "space" definition easier to inspect. NumPy provides the determinant and intersection tests; the combinatorics are intentionally written in plain Python so the logic remains visible.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from itertools import combinations
from math import comb, sqrt
from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Polygon
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display

CHAPTER_NO = 12
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json
from utils.course import chapter_by_no

CHAPTER = chapter_by_no(CHAPTER_NO)
DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = DIRS["figures"]
HTML = DIRS["html"]
CHECKS = DIRS["checks"]
TABLES = DIRS["tables"]

for stale in [FIG / "concept_configuration.svg", FIG / "parameter_experiment.svg", TABLES / "artifact_manifest.csv"]:
    if stale.exists():
        stale.unlink()

COLORS = {
    "ink": "#111827",
    "muted": "#6b7280",
    "blue": "#2563eb",
    "green": "#059669",
    "orange": "#d97706",
    "red": "#dc2626",
    "purple": "#7c3aed",
}

visual_artifacts: list[Path] = []
check_artifacts: list[Path] = []
table_artifacts: list[Path] = []
computed_checks: dict[str, object] = {}

def save_png(fig: plt.Figure, filename: str) -> Path:
    path = FIG / filename
    fig.savefig(path, dpi=165, bbox_inches="tight")
    plt.close(fig)
    visual_artifacts.append(path)
    return path

def display_many(paths: list[Path], width: int = 760) -> None:
    for path in paths:
        display(Markdown(f"### {path.name}"))
        display_artifact(path, width=width)

def rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def orient(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    u = b - a
    v = c - a
    return float(u[0] * v[1] - u[1] * v[0])

def between_scalar(a: float, b: float, c: float) -> bool:
    return min(a, c) < b < max(a, c)


## 1. The Ordered Kernel Inside Euclid

The source span begins by separating two later directions: absolute geometry and affine geometry. Absolute geometry keeps congruence and the first four postulates but does not assume a unique parallel. Affine geometry keeps parallel projection and linear order while discarding metric length and angle. Ordered geometry is the shared nucleus that remains useful to both: points, lines, collinearity, betweenness, sides, rays, and separation.

The first artifact is a concept map rather than a theorem. Its job is to prevent a common misconception: "foundational" does not mean "empty." Betweenness alone can define segments, intervals, rays, line order, sides of a line, angular regions, and many incidence consequences. The later chapters can then add metric, affine, projective, or hyperbolic structure on top of that ordered base.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis("off")

boxes = {
    "ordered": (3.25, 2.25, 3.5, 1.5, "Ordered kernel\\npoints, lines, between, sides"),
    "absolute": (0.65, 4.1, 3.2, 1.1, "Absolute geometry\\ncongruence + order\\nno unique parallel"),
    "affine": (6.15, 4.1, 3.2, 1.1, "Affine geometry\\nparallelism + order\\nno metric angles"),
    "projective": (6.15, 0.75, 3.2, 1.1, "Projective/linear view\\nincidence after adding\\npoints at infinity"),
    "metric": (0.65, 0.75, 3.2, 1.1, "Metric geometry\\nlength, angle, area\\nadded later"),
}
for key, (x, y, w, h, label) in boxes.items():
    face = "#eff6ff" if key == "ordered" else "#f8fafc"
    edge = COLORS["blue"] if key == "ordered" else COLORS["muted"]
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=face, edgecolor=edge, lw=1.8))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=10)

def arrow(start, end, label):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=13, lw=1.5, color=COLORS["ink"]))
    mx, my = (start[0] + end[0]) / 2, (start[1] + end[1]) / 2
    ax.text(mx, my + 0.18, label, ha="center", va="center", fontsize=8, color=COLORS["muted"])

arrow((4.1, 3.8), (2.25, 4.1), "add congruence")
arrow((5.9, 3.8), (7.75, 4.1), "add parallels")
arrow((5.9, 2.25), (7.75, 1.85), "add coordinates")
arrow((4.1, 2.25), (2.25, 1.85), "add measurement")
ax.text(5, 5.7, "Ordered geometry is the common relation layer", ha="center", fontsize=14, weight="bold")

kernel_path = save_png(fig, "euclid-extraction-order-kernel.png")
concept_rows = [
    {"concept": "betweenness", "uses_measurement": False, "supports": "segments, rays, order on a line"},
    {"concept": "side of a line", "uses_measurement": False, "supports": "separation and convex regions"},
    {"concept": "parallelism", "uses_measurement": False, "supports": "affine geometry and ordered ray separation"},
    {"concept": "congruence", "uses_measurement": True, "supports": "absolute and metric geometry"},
]
concept_table = write_csv(TABLES / "ordered-kernel-concepts.csv", concept_rows)
table_artifacts.append(concept_table)
computed_checks["ordered_kernel"] = {"concept_count": len(concept_rows), "source_pdf_pages": CHAPTER["pdf"]}
display_many([kernel_path])


## 2. Intermediacy: Segments, Rays, And Lines

The primitive relation `[ABC]` says that `B` lies between `A` and `C`. From that one relation the chapter defines the segment `AB`, the interval with endpoints included, the ray from `A` away from `B`, and the whole line through `A` and `B`. In the model below, points on a line are represented by real coordinates only so we can compute the relation. The lesson is not that ordered geometry requires coordinates; it is that a coordinate model lets us test the logical behavior of the primitive relation.

Notice the strictness: if `B` is between `A` and `C`, then `A` and `C` must be distinct, and reversing the endpoints keeps the middle point fixed. But `[ABC]` and `[BAC]` cannot both hold. These small facts are the kind of "obvious" statements the chapter insists must be named or proved.

In [ ]:
points_line = {"D": -1.25, "A": 0.0, "E": 0.45, "B": 1.0, "C": 1.85}
triples = [
    ("A", "E", "B"),
    ("B", "E", "A"),
    ("A", "B", "C"),
    ("A", "C", "B"),
    ("D", "A", "C"),
    ("E", "A", "B"),
]
truth_rows = []
for a_name, b_name, c_name in triples:
    truth_rows.append({
        "triple": f"[{a_name}{b_name}{c_name}]",
        "a": points_line[a_name],
        "b": points_line[b_name],
        "c": points_line[c_name],
        "between": between_scalar(points_line[a_name], points_line[b_name], points_line[c_name]),
    })
betweenness_table = write_csv(TABLES / "betweenness-truth-table.csv", truth_rows)
table_artifacts.append(betweenness_table)

fig, axs = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
ax = axs[0]
ax.axhline(0, color=COLORS["ink"], lw=1.2)
ax.plot([points_line["A"], points_line["B"]], [0, 0], color=COLORS["blue"], lw=8, alpha=0.28, solid_capstyle="round", label="segment AB")
for name, x in points_line.items():
    ax.scatter([x], [0], s=70, color=COLORS["orange"] if name in {"A", "B"} else COLORS["green"], zorder=3)
    ax.text(x, 0.14, name, ha="center", fontsize=10)
ax.set_title("betweenness on one line")
ax.set_yticks([])
ax.legend(loc="upper left")
ax.grid(axis="x", alpha=0.25)

ax = axs[1]
ax.axhline(0, color=COLORS["ink"], lw=1.0)
ax.add_patch(FancyArrowPatch((points_line["A"], 0.0), (-1.8, 0.0), arrowstyle="->", mutation_scale=16, lw=2.2, color=COLORS["purple"], label="ray B/A"))
ax.add_patch(FancyArrowPatch((points_line["B"], 0.0), (2.25, 0.0), arrowstyle="->", mutation_scale=16, lw=2.2, color=COLORS["red"], label="ray A/B"))
ax.plot([points_line["A"], points_line["B"]], [0, 0], color=COLORS["blue"], lw=8, alpha=0.25, solid_capstyle="round", label="interval AB")
for name, x in points_line.items():
    ax.scatter([x], [0], s=58, color=COLORS["ink"], zorder=3)
    ax.text(x, 0.14, name, ha="center", fontsize=10)
ax.set_title("line AB decomposes into two rays and the interval")
ax.set_yticks([])
ax.set_xlim(-1.8, 2.25)
ax.grid(axis="x", alpha=0.25)
ax.legend(loc="upper left", ncol=3)

bet_path = save_png(fig, "betweenness-segment-ray-decomposition.png")
computed_checks["intermediacy"] = {
    "true_triples": [row["triple"] for row in truth_rows if row["between"]],
    "false_triples": [row["triple"] for row in truth_rows if not row["between"]],
}
assert between_scalar(points_line["A"], points_line["E"], points_line["B"])
assert between_scalar(points_line["B"], points_line["E"], points_line["A"])
assert not between_scalar(points_line["A"], points_line["C"], points_line["B"])
display_many([bet_path])


## 3. Pasch As A Crossing Invariant

Pasch-style reasoning is the chapter's first major proof engine. A line that passes through a triangle in the right ordered way cannot simply disappear inside the figure. If it enters through one side and exits through another, the third side and the order relations constrain where the intersections may lie.

The figure below is not a copy of the source drawing. It is a computational model of the same logical move. Point `E` is placed between `C` and `A`, while `D` lies beyond `C` on the line through `B` and `C`; the line through them is extended; the code computes its intersection with the remaining side and checks whether each point lies in the relevant open segment. The visual teaches the role of order: the theorem is about sidedness and betweenness, not about measured lengths.

In [ ]:
A = np.array([0.0, 2.2])
B = np.array([-1.8, 0.0])
C = np.array([1.9, 0.0])
D_parameter_on_BC_line = 1.4
D = B + D_parameter_on_BC_line * (C - B)
E = C + 0.42 * (A - C)

def line_intersection(p1, p2, q1, q2):
    r = p2 - p1
    s = q2 - q1
    denom = orient(np.zeros(2), r, s)
    if abs(denom) < 1e-12:
        return None, None, None
    qp = q1 - p1
    t = orient(np.zeros(2), qp, s) / denom
    u = orient(np.zeros(2), qp, r) / denom
    return p1 + t * r, t, u

F, t_de, u_ab = line_intersection(D, E, A, B)

fig, ax = plt.subplots(figsize=(8, 6.8))
tri = np.vstack([A, B, C])
ax.add_patch(Polygon(tri, closed=True, fill=False, edgecolor=COLORS["ink"], lw=2.0))
line_pts = np.vstack([D + 1.25 * (D - E), E + 1.25 * (E - D)])
ax.plot(line_pts[:, 0], line_pts[:, 1], color=COLORS["red"], lw=2.1, label="line DE")
for name, pt, color in [("A", A, COLORS["ink"]), ("B", B, COLORS["ink"]), ("C", C, COLORS["ink"]), ("D", D, COLORS["blue"]), ("E", E, COLORS["blue"]), ("F", F, COLORS["red"])]:
    ax.scatter([pt[0]], [pt[1]], s=70, color=color, zorder=3)
    ax.text(pt[0] + 0.05, pt[1] + 0.06, name, fontsize=11)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Pasch-style crossing: a line through two sides forces ordered intersections")
ax.grid(alpha=0.25)
ax.legend(loc="upper right")
ax.set_xlabel("x")
ax.set_ylabel("y")

pasch_path = save_png(fig, "pasch-triangle-crossing.png")
pasch_checks = {
    "C_between_BD": between_scalar(0.0, 1.0, D_parameter_on_BC_line),
    "E_between_CA": 0 < 0.42 < 1,
    "F_on_segment_AB": 0 < u_ab < 1,
    "line_parameter_for_F": float(t_de),
    "segment_parameter_on_AB": float(u_ab),
}
computed_checks["pasch_crossing"] = pasch_checks
assert pasch_checks["C_between_BD"] and pasch_checks["E_between_CA"] and pasch_checks["F_on_segment_AB"]
display_many([pasch_path])


## 4. Sylvester's Problem: Find An Ordinary Line

The chapter uses ordered geometry to revisit Sylvester's problem of collinear points: in a finite set that is not all collinear, at least one joining line contains exactly two of the points. A proof has to reason about order on a selected line and about which segments are empty. The computational version below does the finite search honestly. It builds every line determined by a pair of points, counts how many of the sample points lie on that line, and highlights ordinary lines.

This does not replace the theorem. A finite search only checks one configuration. Its value is diagnostic: it makes the phrase "line containing exactly two points" inspectable, and it gives the learner a way to test candidate configurations before reading or writing a proof.

In [ ]:
pts = {
    "P1": np.array([0.0, 2.2]),
    "P2": np.array([-2.0, 0.0]),
    "P3": np.array([2.0, 0.0]),
    "P4": np.array([-0.65, 0.0]),
    "P5": np.array([0.95, 0.0]),
    "P6": np.array([0.45, 1.15]),
    "P7": np.array([-1.1, 1.3]),
}
names = list(pts)

def normalized_line(p, q):
    a = q[1] - p[1]
    b = p[0] - q[0]
    c = -(a * p[0] + b * p[1])
    norm = math.hypot(a, b)
    a, b, c = a / norm, b / norm, c / norm
    if a < -1e-12 or (abs(a) < 1e-12 and b < 0):
        a, b, c = -a, -b, -c
    return np.array([a, b, c])

line_records = []
seen = {}
for i, j in combinations(names, 2):
    line = normalized_line(pts[i], pts[j])
    key = tuple(np.round(line, 10))
    if key in seen:
        continue
    on_line = [name for name in names if abs(line[0] * pts[name][0] + line[1] * pts[name][1] + line[2]) < 1e-9]
    seen[key] = True
    line_records.append({"line": line, "points": on_line, "count": len(on_line)})
ordinary = [record for record in line_records if record["count"] == 2]

fig, ax = plt.subplots(figsize=(8.2, 6.4))
for record in ordinary[:10]:
    p_name, q_name = record["points"]
    p, q = pts[p_name], pts[q_name]
    direction = q - p
    direction = direction / np.linalg.norm(direction)
    center = 0.5 * (p + q)
    segment = np.vstack([center - 3.2 * direction, center + 3.2 * direction])
    ax.plot(segment[:, 0], segment[:, 1], color=COLORS["green"], lw=1.0, alpha=0.35)
if ordinary:
    p_name, q_name = ordinary[0]["points"]
    p, q = pts[p_name], pts[q_name]
    direction = (q - p) / np.linalg.norm(q - p)
    center = 0.5 * (p + q)
    segment = np.vstack([center - 3.2 * direction, center + 3.2 * direction])
    ax.plot(segment[:, 0], segment[:, 1], color=COLORS["red"], lw=2.4, label=f"ordinary line {p_name}{q_name}")
for name, point in pts.items():
    ax.scatter([point[0]], [point[1]], s=68, color=COLORS["blue"], zorder=3)
    ax.text(point[0] + 0.05, point[1] + 0.06, name, fontsize=10)
ax.set_title("Sylvester search: ordinary lines in a non-collinear finite set")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.legend(loc="upper right")
ax.set_xlabel("x")
ax.set_ylabel("y")

sylvester_path = save_png(fig, "sylvester-ordinary-line-search.png")
ordinary_rows = [
    {"points_on_line": " ".join(record["points"]), "count": record["count"]}
    for record in line_records
]
ordinary_table = write_csv(TABLES / "ordinary-line-search.csv", ordinary_rows)
table_artifacts.append(ordinary_table)
computed_checks["sylvester"] = {
    "point_count": len(pts),
    "distinct_joining_lines": len(line_records),
    "ordinary_line_count": len(ordinary),
    "first_ordinary_line": ordinary[0]["points"] if ordinary else [],
}
assert len(ordinary) > 0
assert any(record["count"] > 2 for record in line_records)
display_many([sylvester_path])


## 5. Planes, Hyperplanes, And Regions

The chapter delays the word "plane" until order has already done a surprising amount of work. Once planes and higher-dimensional spaces are introduced, hyperplanes decompose space into convex regions. In general position, `m` hyperplanes in `n` dimensions cut space into `sum_{k=0}^{n} binom(m,k)` regions. This formula is a compact way to visualize the region-count exercises in the source span.

The static graph records the counts for lines in a plane and planes in space. The Plotly tetrahedron is a separate incidence model: four non-coplanar points define a 3-space, and faces/edges make it possible to track where a line through the space meets the simplex. Rotate it to see why a flat page is a poor substitute for this part of the chapter.

In [ ]:
def region_count(dim: int, hyperplanes: int) -> int:
    return sum(comb(hyperplanes, k) for k in range(0, min(dim, hyperplanes) + 1))

rows = []
for dim in [1, 2, 3, 4]:
    for m in range(0, 9):
        rows.append({"dimension": dim, "hyperplanes": m, "regions": region_count(dim, m)})
regions_table = write_csv(TABLES / "hyperplane-region-counts.csv", rows)
table_artifacts.append(regions_table)

recurrence_ok = True
for dim in [1, 2, 3, 4]:
    for m in range(1, 9):
        recurrence_ok &= region_count(dim, m) == region_count(dim, m - 1) + region_count(dim - 1, m - 1)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
for dim, color in [(1, COLORS["muted"]), (2, COLORS["blue"]), (3, COLORS["orange"]), (4, COLORS["purple"])]:
    xs = np.arange(0, 9)
    ys = [region_count(dim, int(m)) for m in xs]
    ax.plot(xs, ys, marker="o", lw=2.2, color=color, label=f"n={dim}")
ax.set_title("Maximum regions cut out by hyperplanes in general position")
ax.set_xlabel("number of hyperplanes")
ax.set_ylabel("regions")
ax.grid(alpha=0.25)
ax.legend(title="dimension")
regions_path = save_png(fig, "hyperplane-region-counts.png")

vertices = {
    "A": np.array([0.0, 0.0, 0.0]),
    "B": np.array([1.2, 0.0, 0.0]),
    "C": np.array([0.35, 1.0, 0.0]),
    "D": np.array([0.35, 0.35, 1.1]),
}
faces = [("A", "B", "C"), ("A", "B", "D"), ("A", "C", "D"), ("B", "C", "D")]
fig3 = go.Figure()
for face in faces:
    xyz = np.array([vertices[name] for name in face])
    fig3.add_trace(go.Mesh3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], color="#93c5fd", opacity=0.28, showscale=False, name="face " + "".join(face)))
for i, j in combinations(vertices, 2):
    segment = np.vstack([vertices[i], vertices[j]])
    fig3.add_trace(go.Scatter3d(x=segment[:, 0], y=segment[:, 1], z=segment[:, 2], mode="lines", line=dict(color="#111827", width=4), showlegend=False))
for name, point in vertices.items():
    fig3.add_trace(go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode="markers+text", text=[name], textposition="top center", marker=dict(size=5, color="#dc2626"), showlegend=False))
P = 0.25 * vertices["A"] + 0.35 * vertices["B"] + 0.40 * vertices["C"]
Q = np.array([1.25, 0.75, 0.95])
line = np.vstack([P, Q])
fig3.add_trace(go.Scatter3d(x=line[:, 0], y=line[:, 1], z=line[:, 2], mode="lines+markers", line=dict(color="#7c3aed", width=5), marker=dict(size=4), name="line through face point"))
fig3.update_layout(title="Simplex space incidence model", scene=dict(aspectmode="data"), width=820, height=620, template="plotly_white")
simplex_path = HTML / "simplex-space-incidence.html"
if simplex_path.exists():
    simplex_path.unlink()
fig3.write_html(simplex_path, include_plotlyjs="cdn", full_html=True)
visual_artifacts.append(simplex_path)

computed_checks["planes_hyperplanes"] = {
    "recurrence_ok": bool(recurrence_ok),
    "f_2_3": region_count(2, 3),
    "f_3_4": region_count(3, 4),
    "simplex_vertices": len(vertices),
    "simplex_faces": len(faces),
}
assert recurrence_ok
assert region_count(2, 3) == 7
assert region_count(3, 4) == 15
display_many([regions_path, simplex_path])


## 6. Continuity As A Dedekind Cut

Order alone gives infinitely many points between two points, but that is not yet continuity. The source span emphasizes the gap between "there is always another point" and "there is no missing boundary." A Dedekind cut captures the stronger idea by partitioning a line into two nonempty parts with all of one side before all of the other. For the square-root example, rationals with square less than `2` approach a boundary that is not rational.

The figure uses rational samples only. The lower and upper rational bounds improve as denominators grow, but the boundary itself is marked by `sqrt(2)`. This is the computational lesson: density gives endless approximation; continuity supplies the limiting point.

In [ ]:
samples = []
for q in range(1, 80):
    for p in range(1, 3 * q):
        x = p / q
        if 1.0 <= x <= 1.7:
            samples.append((p, q, x, x * x < 2))
lower = max(x for _, _, x, is_lower in samples if is_lower)
upper = min(x for _, _, x, is_lower in samples if not is_lower)
sqrt2 = sqrt(2)

fig, ax = plt.subplots(figsize=(11, 4.2))
for p, q, x, is_lower in samples:
    y = 1 / q
    ax.scatter([x], [y], s=13, color=COLORS["blue"] if is_lower else COLORS["orange"], alpha=0.45)
ax.axvline(sqrt2, color=COLORS["red"], lw=2.2, label="sqrt(2)")
ax.axvspan(lower, upper, color=COLORS["red"], alpha=0.12, label="narrowest sampled rational bracket")
ax.set_title("Dedekind cut: rational samples approach a non-rational boundary")
ax.set_xlabel("rational sample x")
ax.set_ylabel("1 / denominator")
ax.grid(alpha=0.25)
ax.legend(loc="upper right")

cut_path = save_png(fig, "dedekind-cut-continuity.png")
cut_rows = [
    {"side": "lower", "best_rational": lower, "distance_to_sqrt2": sqrt2 - lower},
    {"side": "upper", "best_rational": upper, "distance_to_sqrt2": upper - sqrt2},
]
cut_table = write_csv(TABLES / "dedekind-cut-bracket.csv", cut_rows)
table_artifacts.append(cut_table)
computed_checks["continuity"] = {
    "sample_count": len(samples),
    "best_lower": lower,
    "best_upper": upper,
    "contains_sqrt2": lower < sqrt2 < upper,
    "bracket_width": upper - lower,
}
assert lower < sqrt2 < upper
display_many([cut_path])


## 7. Parallelism As Ray Separation

The chapter's last section treats parallelism without using angle measure. Through a point not on a line, ordered geometry can distinguish rays that meet the line from rays that do not, and the boundary directions separate those two behaviors. In Euclidean coordinates the boundary rays are the two opposite directions along the line through `A` parallel to `r`; in hyperbolic settings the analogous boundary rays are more subtle, which is why the later hyperbolic chapters return to this topic.

The figure below is deliberately Euclidean because it keeps the classification transparent. A ray from `A` intersects the horizontal line `r` exactly when its vertical component points downward. The two horizontal rays do not meet `r` and form the boundary between the downward meeting fan and the upward nonmeeting fan. The code records the classification rather than relying on the eye.

In [ ]:
A0 = np.array([0.0, 1.25])
angles = np.linspace(-165, 165, 23) * np.pi / 180
ray_rows = []
fig, ax = plt.subplots(figsize=(10, 6.2))
ax.axhline(0, color=COLORS["ink"], lw=2.0, label="line r")
ax.scatter([A0[0]], [A0[1]], s=80, color=COLORS["red"], zorder=4)
ax.text(A0[0] + 0.05, A0[1] + 0.06, "A", fontsize=11)
for theta in angles:
    direction = np.array([math.cos(theta), math.sin(theta)])
    meets = direction[1] < -1e-9
    boundary = abs(direction[1]) <= 1e-9
    color = COLORS["green"] if meets else COLORS["orange"]
    if boundary:
        color = COLORS["purple"]
    end = A0 + 1.75 * direction
    ax.add_patch(FancyArrowPatch(A0, end, arrowstyle="->", mutation_scale=10, lw=1.5 if not boundary else 2.4, color=color, alpha=0.72))
    if meets:
        t_hit = -A0[1] / direction[1]
        hit = A0 + t_hit * direction
        ax.scatter([hit[0]], [0], s=22, color=COLORS["green"], alpha=0.7)
    else:
        t_hit = None
    ray_rows.append({"angle_degrees": float(theta * 180 / np.pi), "meets_line_r": bool(meets), "boundary_parallel": bool(boundary), "hit_parameter": None if t_hit is None else float(t_hit)})
ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-0.65, 2.25)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Parallel directions separate rays that meet r from rays that do not")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.25)
handles = [
    plt.Line2D([0], [0], color=COLORS["green"], lw=2, label="meets r"),
    plt.Line2D([0], [0], color=COLORS["orange"], lw=2, label="does not meet r"),
    plt.Line2D([0], [0], color=COLORS["purple"], lw=2, label="parallel boundary"),
]
ax.legend(handles=handles, loc="upper right")

parallel_path = save_png(fig, "parallelism-separating-rays.png")
parallel_table = write_csv(TABLES / "parallel-ray-classification.csv", ray_rows)
table_artifacts.append(parallel_table)
computed_checks["parallelism"] = {
    "sampled_rays": len(ray_rows),
    "meeting_rays": sum(row["meets_line_r"] for row in ray_rows),
    "nonmeeting_rays": sum(not row["meets_line_r"] for row in ray_rows),
    "boundary_rays": sum(row["boundary_parallel"] for row in ray_rows),
}
assert computed_checks["parallelism"]["meeting_rays"] > 0
assert computed_checks["parallelism"]["nonmeeting_rays"] > 0
display_many([parallel_path])


## What The Checks Prove And What They Do Not

The models above are not axioms. A coordinate model can satisfy an ordered axiom and still fail to explain why the axiom is independent, minimal, or historically necessary. What the models do well is keep the primitive relation visible. Betweenness becomes a predicate with allowed and forbidden orders. Pasch becomes a crossing invariant. Sylvester's theorem becomes a finite search target. Hyperplane regions become a recurrence. Continuity becomes the difference between density and completion. Parallelism becomes a separation of rays.

That makes the chapter stand on its own: the reader can manipulate examples, rerun checks, and see exactly which ordered claim is being inspected before later chapters add affine coordinates, congruence, projective closure, or hyperbolic models.

In [ ]:
manifest_rows = []
for artifact in visual_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "visual", "concept": artifact.stem.replace("-", " ")})
for artifact in table_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "table", "concept": artifact.stem.replace("-", " ")})
manifest_path = write_csv(TABLES / "artifact-manifest.csv", manifest_rows)
table_artifacts.append(manifest_path)

summary_payload = {
    "chapter": CHAPTER_NO,
    "title": CHAPTER["title"],
    "source_printed_pages": CHAPTER["printed"],
    "source_pdf_pages": CHAPTER["pdf"],
    "visuals": [rel(path) for path in visual_artifacts],
    "tables": [rel(path) for path in table_artifacts],
    "checks": computed_checks,
}
summary_path = write_json(CHECKS / "visual-summary.json", summary_payload)
legacy_summary_path = write_json(CHECKS / "visual_summary.json", summary_payload)
check_artifacts.extend([summary_path, legacy_summary_path])

final_sanity = {
    "visual_count": len(visual_artifacts),
    "table_count": len(table_artifacts),
    "pasch_F_on_segment_AB": computed_checks["pasch_crossing"]["F_on_segment_AB"],
    "ordinary_line_count": computed_checks["sylvester"]["ordinary_line_count"],
    "hyperplane_recurrence_ok": computed_checks["planes_hyperplanes"]["recurrence_ok"],
    "dedekind_contains_sqrt2": computed_checks["continuity"]["contains_sqrt2"],
    "parallel_boundary_rays": computed_checks["parallelism"]["boundary_rays"],
}
final_sanity_path = write_json(CHECKS / "final-sanity-summary.json", final_sanity)
check_artifacts.append(final_sanity_path)

assert_artifacts(visual_artifacts + table_artifacts + check_artifacts, min_bytes=80)
assert final_sanity["visual_count"] >= 7
assert final_sanity["pasch_F_on_segment_AB"]
assert final_sanity["ordinary_line_count"] > 0
assert final_sanity["hyperplane_recurrence_ok"]
assert final_sanity["dedekind_contains_sqrt2"]
assert final_sanity["parallel_boundary_rays"] >= 1
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()
assert not (TABLES / "artifact_manifest.csv").exists()

summary_payload


## Takeaways

- Ordered geometry keeps incidence, betweenness, separation, sides, segments, rays, and lines before adding measurement.
- The relation `[ABC]` is powerful enough to define segments and rays, but its symmetry and exclusion rules must be stated carefully.
- Pasch-style reasoning is about crossings and sides: a line cannot enter a triangle through ordered sides without constrained exits.
- Sylvester's problem turns collinearity into a finite search for ordinary lines and shows how order supports nontrivial incidence proofs.
- Planes and hyperplanes extend the same separation idea into higher dimensions; region counts are a compact computational check.
- Continuity is stronger than density. Rational points can approach a cut forever without supplying the boundary point.
- Parallelism can be read as a ray-separation statement, which prepares the later distinction between affine, absolute, and hyperbolic geometry.